## Système de reconnaissance faciale — Notebook unique, commenté (moteur IA + MongoDB + API Flask)

Ce notebook contient tout le pipeline : chargement des modèles InsightFace, communication avec MongoDB,
et l'API Flask qui sert le flux vidéo (MJPEG) et les pages du tableau de bord.

Chaque section est précédée d'une cellule Markdown expliquant : **le rôle** de la cellule, **son fonctionnement**,
les **principes mathématiques** sous-jacents quand il y en a, et l'**origine/histoire** des technologies utilisées.

**Ordre d'exécution important** : toutes les cellules doivent être exécutées dans l'ordre, de haut en bas,
avant la dernière cellule (`app.run(...)`) qui démarre le serveur et bloque le kernel.

Pour arrêter le serveur : interrompre le kernel. Pour relancer après une modification de code : *Run All*.

**Vue d'ensemble des technologies utilisées dans ce projet :** Flask (serveur web), MongoDB/pymongo (base de données), InsightFace + ONNX Runtime (détection SCRFD et reconnaissance ArcFace), OpenCV (capture caméra, traitement d'image), NumPy (calcul vectoriel), Jinja2 (templates HTML), Werkzeug (sécurité des mots de passe, serveur de développement), psutil (métriques système).

### 1. Import de Flask — le framework web

**Définition** : Flask est un micro-framework web en Python — une bibliothèque légère qui fournit l'essentiel pour créer un serveur web (routage des URL, gestion des requêtes/réponses HTTP) sans imposer de structure lourde comme le font des frameworks plus complets (Django, par exemple).

**Rôle** : Flask fournit le cœur du serveur : associer une URL à une fonction Python (le *routage*), construire les
réponses HTTP, et faire le lien avec les fichiers HTML via un moteur de templates.

**Origine et histoire** : Flask a été créé en 2010 par le développeur allemand Armin Ronacher, à l'origine comme un
prototype publié le 1er avril (un « poisson d'avril » technique). L'idée était de proposer une alternative légère à
Django (2005), qui impose beaucoup de structure (ORM, admin, authentification intégrés). Flask est un
*micro-framework* : il ne fournit que le strict nécessaire et laisse le développeur choisir ses propres outils.
Il s'appuie sur deux autres bibliothèques que Ronacher avait déjà écrites :
- **Werkzeug**, une boîte à outils WSGI (le protocole bas niveau entre un serveur web et une application Python)
- **Jinja2**, le moteur de templates qui permettra d'injecter des variables Python dans les fichiers `.html`

**Fonctionnement** : chaque nom importé a un rôle précis dans ce notebook :
- `Flask` : la classe application, instanciée une seule fois (`app = Flask(__name__)`, cellule 16)
- `Response` : construit une réponse HTTP « sur mesure » — indispensable pour le flux vidéo, qui n'est pas du HTML
- `render_template` : charge un fichier de `templates/` et y remplace les `{{ variable }}` par des valeurs Python
- `request` : représente la requête HTTP entrante (on l'utilisera pour lire le formulaire d'identification)
- `redirect`, `url_for` : gèrent la redirection HTTP (code 302) et génèrent une URL à partir du *nom* d'une route
  plutôt que de l'écrire en dur — si le chemin d'une route change un jour, les liens ne cassent pas

**Outils et technologies :** Flask (framework web WSGI), et via lui Werkzeug (serveur HTTP) et Jinja2 (moteur de templates).

### 2. Imports de la chaîne de vision par ordinateur

**Définition** : cette section importe trois familles d'outils distinctes, chacune couvrant une étape du pipeline de vision par ordinateur : OpenCV pour capturer et manipuler les images/flux vidéo, InsightFace (via ONNX Runtime) pour la détection et la reconnaissance de visages par réseaux de neurones, et NumPy pour manipuler efficacement les tableaux de nombres (pixels, embeddings) sous-jacents.

**Rôle** : ces bibliothèques fournissent la capture/manipulation d'images (`cv2`), le calcul numérique vectoriel
(`numpy`), la recherche de fichiers (`glob`), et l'accès aux modèles de reconnaissance faciale (`insightface`).

**Origine et histoire** :
- **OpenCV** (`cv2`) a été lancé en 1999 par Gary Bradski chez Intel. Le but initial était presque publicitaire :
  démontrer des applications gourmandes en calcul pour stimuler la vente de processeurs Intel plus puissants.
  Le projet est devenu la bibliothèque de vision par ordinateur open source la plus utilisée au monde.
- **NumPy** a été créé en 2005 par Travis Oliphant, en fusionnant deux projets antérieurs concurrents
  (*Numeric*, 1995, et *Numarray*). Il introduit la structure `ndarray` (tableau multidimensionnel), fondation de
  quasiment tout l'écosystème scientifique Python (pandas, scikit-learn, PyTorch s'en inspirent ou l'utilisent).
- **InsightFace** est un projet de recherche open source (organisation *deepinsight*) né vers 2018, qui regroupe
  plusieurs travaux publiés par ses auteurs sur la détection et la reconnaissance faciale (ArcFace, RetinaFace,
  SCRFD — détaillés dans les cellules 6-7). `Face` est une simple structure de données (un conteneur) qui regroupe
  la boîte englobante (`bbox`), les points de repère du visage (`kps`, *keypoints* : yeux, nez, coins de la bouche),
  et plus tard l'embedding calculé. `model_zoo` est l'utilitaire qui télécharge et charge les poids pré-entraînés
  au format ONNX (*Open Neural Network Exchange*, format d'échange de modèles créé par Microsoft et Facebook en
  2017 pour rendre les modèles interopérables entre frameworks).

**Outils et technologies :** OpenCV (`cv2`, capture caméra et traitement d'image), NumPy (calcul vectoriel), InsightFace (détection/reconnaissance faciale) et ONNX Runtime (moteur d'inférence sous-jacent, via `model_zoo`), `glob` (recherche de fichiers, bibliothèque standard).

### 3. Imports MongoDB et utilitaires système

**Définition** : MongoDB est une base de données NoSQL orientée documents — elle stocke des enregistrements sous forme de documents JSON flexibles (plutôt que des lignes de tableau à colonnes fixes comme en SQL), ce qui convient bien ici puisque chaque type de fiche (détection, personne, inconnu) n'a pas exactement la même forme. `pymongo` est la bibliothèque Python qui permet de s'y connecter et d'y lire/écrire.

**Rôle** : `pymongo` est le pilote officiel permettant à Python de dialoguer avec MongoDB. `datetime` fournit les
horodatages des détections. `os` gère les chemins de fichiers et la création de dossiers. `json` sert à lire/écrire
les fichiers de configuration persistés sur disque (`data/cameras.json`, `data/admin.json`, `acces.json`).

**Origine** : `pymongo` est maintenu directement par MongoDB Inc. depuis la création de la base en 2009. `datetime`
et `os` font partie de la bibliothèque standard de Python depuis ses toutes premières versions (1991).

**Outils et technologies :** pymongo (client MongoDB), et plusieurs modules de la bibliothèque standard Python : `datetime`/`timedelta` (horodatage), `os` et `json` (fichiers de configuration).

### 4. Configuration générale

**Définition** : ce bloc ne définit pas une fonction mais un ensemble de constantes de configuration — des valeurs fixes (seuils, chemins de dossiers, noms de fichiers) utilisées à plusieurs endroits du programme, regroupées ici pour être modifiables à un seul endroit plutôt que dispersées dans le code.

**Rôle** : centraliser les constantes du projet — dossiers de sauvegarde des visages, seuil de décision, et la
correspondance entre le nom logique d'une caméra et sa source réelle.

**Principe mathématique — le seuil `SEUIL_DEFAUT`** : ce n'est pas une constante arbitraire, c'est un curseur qui
règle un compromis statistique classique en biométrie, entre deux types d'erreurs :
- le **FAR** (*False Acceptance Rate*) : accepter à tort un inconnu comme une personne connue (seuil trop bas)
- le **FRR** (*False Rejection Rate*) : rejeter à tort une personne connue comme inconnue (seuil trop haut)

En traçant FAR et FRR en fonction du seuil, leur point de croisement s'appelle l'**EER** (*Equal Error Rate*) —
une métrique standard pour évaluer un système biométrique. `0.5` est une valeur de départ raisonnable pour des
embeddings ArcFace normalisés, à ajuster empiriquement selon vos données réelles.

**Origine — DroidCam** : DroidCam est une application créée par la société Dev47Apps, qui transforme un smartphone
en webcam, exposée soit comme périphérique virtuel local (connexion USB, via un pilote/« client » installé sur le
PC), soit comme un flux HTTP accessible en réseau local (Wi-Fi, par défaut sur le port `4747`, au format MJPEG —
voir cellule 16 pour le détail de ce format).

**Mise à jour** : les caméras sont maintenant persistées dans `data/cameras.json`, pour survivre à un redémarrage du kernel (ajout/suppression via la page Paramètres).

**Outils et technologies :** aucune bibliothèque externe ici — uniquement des constantes Python et la bibliothèque standard (`os`, `json`) pour lire/écrire `data/cameras.json`.

### 5. Connexion à MongoDB

**Définition** : se "connecter" à MongoDB signifie établir une communication réseau entre le programme Python et le serveur de base de données (ici en local ou via une URI Atlas), à travers laquelle toutes les lectures/écritures suivantes transiteront.

**Rôle** : ouvrir la connexion vers le serveur MongoDB local et vérifier immédiatement qu'elle fonctionne.

**Origine et histoire** : MongoDB a été créé en 2007 par Dwight Merriman, Eliot Horowitz et Kevin Ryan, sous le nom
de société *10gen* (renommée MongoDB Inc. en 2013). Le nom « Mongo » vient de « humongous » (énorme) — l'objectif
initial était de gérer de très gros volumes de données avec un modèle plus flexible que les bases relationnelles
classiques (SQL). MongoDB appartient à la famille **NoSQL**, plus précisément aux bases *orientées documents* :
au lieu de lignes dans des tables à colonnes fixes, elle stocke des documents au format **BSON** (*Binary JSON*),
une extension binaire de JSON qui ajoute des types que JSON ne supporte pas nativement (dates, identifiants
binaires `ObjectId`, données binaires brutes). C'est ce qui permet ici de stocker un embedding (une liste de 512
nombres) directement comme un champ de document, sans schéma de table à définir à l'avance.

**Fonctionnement** : `MongoClient(...)` ouvre une connexion *paresseuse* (« lazy ») — elle ne vérifie rien tout de
suite. `client.admin.command("ping")` envoie une commande d'administration minimale (faisant partie du protocole
réseau natif de MongoDB, le *wire protocol*) pour forcer une vérification immédiate plutôt que de découvrir un
problème de connexion plus tard, au milieu d'une requête plus complexe.

**Outils et technologies :** pymongo (`MongoClient`), qui communique avec le serveur MongoDB via son propre protocole binaire (*MongoDB Wire Protocol*).

### 6. Chargement du modèle de détection (`det_10g.onnx` — SCRFD)

**Définition** : SCRFD est un réseau de neurones spécialisé dans la **détection** de visages — sa tâche est uniquement de trouver *où* se trouvent des visages dans une image (en dessinant une boîte autour de chacun), pas de dire *qui* ils sont. Il est ici chargé au format ONNX, un format standard qui permet d'exécuter un modèle entraîné sans avoir besoin de l'outil d'entraînement d'origine.

**Rôle** : ce modèle repère *où* se trouvent les visages dans une image — il ne les identifie pas, il les localise.
Pour chaque visage trouvé, il retourne une boîte englobante (`bbox`) et 5 points de repère (`kps` : les deux yeux,
le nez, les deux coins de la bouche).

**Origine et histoire** : `det_10g.onnx` implémente **SCRFD** (*Sample and Computation Redistribution for Efficient
Face Detection*), publié en 2021 par l'équipe InsightFace (Guo, Deng et al.). SCRFD s'inscrit dans une lignée de
détecteurs mono-étape (« single-stage ») remontant à **RetinaNet** (Facebook AI Research, 2017 — qui a introduit la
*focal loss* pour compenser le déséquilibre entre zones de fond et zones contenant un objet) et aux **FPN**
(*Feature Pyramid Networks*, 2016), qui permettent de détecter des visages à plusieurs échelles simultanément.
**RetinaFace** (Deng et al., 2019) a adapté cette architecture spécifiquement aux visages, en ajoutant la
régression des 5 points de repère en plus de la boîte. SCRFD est ensuite venu optimiser le rapport précision/vitesse
de RetinaFace en redistribuant intelligemment le calcul entre les différentes échelles du réseau.

**Principe mathématique** : comme la plupart des détecteurs modernes, le réseau ne prédit pas directement des
coordonnées absolues. Il définit une grille de points *ancres* sur l'image, et pour chacun prédit :
- un score de confiance (visage / pas visage), via une fonction sigmoïde
- un décalage `(dx, dy, dw, dh)` par rapport à une boîte de référence — principe de régression de boîte introduit
  par R-CNN et ses successeurs (2014-2015)

Comme un même visage peut être détecté par plusieurs ancres voisines, un post-traitement appelé **NMS**
(*Non-Maximum Suppression*) élimine les doublons : il garde la détection au score le plus élevé et supprime toute
autre boîte dont le recouvrement avec elle (mesuré par l'**IoU**, *Intersection over Union* — l'aire d'intersection
divisée par l'aire d'union des deux boîtes) dépasse un certain seuil. `max_num=0` dans nos appels signifie
« aucune limite sur le nombre de visages détectés ».

**Outils et technologies :** InsightFace (`model_zoo.get_model`), ONNX Runtime (moteur qui exécute réellement le réseau de neurones SCRFD au format `.onnx`).

### 7. Chargement du modèle de reconnaissance (`w600k_r50.onnx` — ArcFace)

**Définition** : ArcFace est un réseau de neurones spécialisé dans la **reconnaissance** faciale — contrairement à SCRFD (qui détecte), sa tâche est de transformer un visage déjà localisé en un vecteur numérique (l'*embedding*, voir section 8) qui résume ses traits de façon comparable à ceux d'un autre visage.

**Rôle** : contrairement au modèle précédent qui *localise*, celui-ci *caractérise* — il transforme un visage déjà
détecté et aligné en un vecteur numérique de 512 dimensions (l'**embedding**), une sorte d'empreinte mathématique
du visage. Deux photos de la même personne doivent produire deux vecteurs proches ; deux personnes différentes,
deux vecteurs éloignés.

**Origine et histoire** : le nom du fichier indique deux choses : `r50` = une architecture **ResNet-50**
(*Residual Network*, He et al., Microsoft Research, 2015 — l'article qui a introduit les *connexions résiduelles*,
permettant d'entraîner des réseaux beaucoup plus profonds en laissant le signal du gradient « sauter » certaines
couches pendant l'apprentissage, ce qui a résolu le problème du gradient qui s'évanouit dans les réseaux profonds).
`w600k` fait référence au jeu de données d'entraînement (dérivé de MS1M/Glint360k, plusieurs millions d'images
couvrant des centaines de milliers d'identités).

Le réseau est entraîné avec la fonction de perte **ArcFace** (*Additive Angular Margin Loss*, Deng, Guo, Xue,
Zafeiriou — 2019, InsightFace). C'est l'élément le plus important à comprendre mathématiquement :

**Principe mathématique — ArcFace** : un classifieur softmax classique compare un vecteur caractéristique à des
vecteurs de référence via leur produit scalaire, ce qui revient (une fois les vecteurs normalisés) à comparer des
**cosinus d'angles**. ArcFace ajoute une marge angulaire `m` directement à l'angle θ entre le vecteur et sa classe
correcte, avant de recalculer le cosinus : la perte optimise `cos(θ + m)` plutôt que `cos(θ)`. Concrètement, le
réseau est *forcé* pendant l'entraînement à rapprocher angulairement les visages d'une même personne beaucoup plus
qu'un entraînement softmax classique ne l'exigerait, et à repousser les personnes différentes plus loin sur la
sphère. Résultat : les embeddings d'une même identité se regroupent en un cône angulaire étroit, ce qui rend la
simple distance angulaire (donc la similarité cosinus, cellule 8) directement utilisable comme mesure de
reconnaissance — sans réseau de comparaison supplémentaire.

`face.normed_embedding` (utilisé plus loin) est déjà **normalisé** (norme euclidienne = 1) : tous les embeddings
vivent sur une sphère unité de dimension 512, ce qui est précisément l'hypothèse dont a besoin la similarité
cosinus pour être un simple produit scalaire (cellule 8).

**Outils et technologies :** InsightFace (`model_zoo.get_model`), ONNX Runtime (exécution du réseau ArcFace `w600k_r50.onnx`).

### 8. `find_match()` — comparaison par similarité cosinus

**Définition** : la similarité cosinus est une mesure mathématique de à quel point deux vecteurs "pointent dans la même direction", indépendamment de leur longueur — une valeur proche de 1 signifie qu'ils sont presque identiques en orientation, proche de 0 qu'ils sont indépendants. C'est la mesure standard pour comparer deux embeddings faciaux.

**Rôle** : comparer l'embedding d'un visage détecté à tous les embeddings connus en base, et retourner le nom le
plus proche — ou `"Inconnu"` si même le plus proche reste trop loin.

**Principe mathématique — similarité cosinus** : pour deux vecteurs normalisés **a** et **b** (norme = 1), leur
produit scalaire `a · b = Σ aᵢbᵢ` est *exactement égal* à `cos(θ)`, où θ est l'angle entre les deux vecteurs. C'est
une conséquence directe de la définition géométrique du produit scalaire : `a · b = ‖a‖‖b‖cos(θ)`, qui se simplifie
ici puisque `‖a‖ = ‖b‖ = 1`. Une valeur de 1 signifie des vecteurs identiques en direction (angle nul), 0 signifie
des vecteurs orthogonaux (aucune corrélation), et -1 des vecteurs opposés. `np.dot(embedding, known_embeddings.T)`
calcule ce produit scalaire simultanément contre *tous* les embeddings connus (une multiplication matrice-vecteur),
ce qui est bien plus rapide qu'une boucle Python comparant un par un.

`np.argmax(scores)` sélectionne l'indice du score le plus élevé — c'est un classifieur du **plus proche voisin**
(*1-nearest neighbor*, ou *1-NN*), l'une des méthodes de classification les plus anciennes et les plus simples en
apprentissage automatique (formalisée par Cover et Hart en 1967 dans leur article fondateur sur la classification
par plus proche voisin). Le concept même de « similarité cosinus » pour comparer des vecteurs vient historiquement
du **modèle vectoriel** en recherche d'information (Gerard Salton, années 1970), où l'on représentait des documents
texte comme des vecteurs pour mesurer leur ressemblance — la même idée mathématique est réutilisée ici, appliquée
non pas à des mots mais aux embeddings faciaux.

Le `threshold` est le curseur FAR/FRR décrit en cellule 4 : sous ce seuil, même la meilleure correspondance n'est
pas jugée assez fiable et le visage est classé `"Inconnu"`.

**Outils et technologies :** NumPy (`np.dot` pour le produit scalaire, `np.argmax` pour trouver le meilleur score).

### 9. `agrandir_bbox_epaules()` — recadrage géométrique

**Définition** : une *bounding box* (boîte englobante, abrégée *bbox*) est le rectangle qui délimite un objet détecté dans une image, généralement représenté par les coordonnées de ses coins (x1, y1, x2, y2).

**Rôle** : cette fonction n'a **aucun lien avec l'IA** — c'est de la géométrie pure. Elle agrandit la boîte du
visage détecté pour inclure les épaules, uniquement pour que l'image *sauvegardée sur disque* (dans `inconnus/`
ou `succes/`) soit plus lisible pour un humain qui la consulterait plus tard. L'embedding, lui, a déjà été calculé
à l'étape précédente sur le visage seul — cette fonction n'intervient jamais dans la reconnaissance elle-même.

**Fonctionnement mathématique** : chaque marge (`marge_haut`, `marge_bas`, `marge_cotes`) est un pourcentage de la
largeur ou hauteur de la boîte d'origine, ajouté de chaque côté, puis limité (`max(0, ...)` / `min(largeur_frame,
...)`) pour ne jamais sortir des limites de l'image. C'est une simple opération d'échelle et de translation,
sans aucun modèle statistique derrière — les valeurs par défaut (0.5, 0.8, 0.6) ont été choisies empiriquement pour
englober les épaules sans capturer trop d'arrière-plan.

**Outils et technologies :** Python pur (arithmétique de coordonnées) — aucune bibliothèque externe requise pour ce calcul géométrique.

### 10. `charger_embeddings_mongo()` — lecture de la base connue

**Définition** : un *embedding* est le vecteur numérique produit par ArcFace (section 7) pour représenter un visage. "Charger les embeddings connus" signifie relire depuis MongoDB tous les vecteurs déjà enregistrés pour les personnes autorisées, afin de pouvoir comparer chaque nouveau visage détecté à cette base.

**Rôle** : recharger en mémoire tous les embeddings enregistrés, sous la forme attendue par `find_match()` (une
matrice NumPy + une liste de noms parallèle).

**Fonctionnement** : `db.Persons.find({}, {"nom": 1, "embedding": 1})` est une requête MongoDB : le
premier argument `{}` signifie « aucun filtre, tous les documents », le second est une *projection* qui limite les
champs renvoyés (économie de bande passante — inutile de rapatrier `role`/`departement` ici). Le langage de requête
de MongoDB — des documents JSON décrivant le filtre — a été conçu délibérément pour ressembler à la syntaxe des
objets JavaScript, MongoDB ayant historiquement ciblé en priorité les développeurs web (Node.js) dans son adoption
initiale à la fin des années 2000.

`np.array([d["embedding"] for d in docs])` empile toutes les listes de 512 nombres en une seule matrice de forme
`(nombre_de_visages, 512)` — c'est cette matrice que `find_match()` utilise pour comparer un visage à tous les
autres en une seule opération matricielle plutôt qu'une boucle.

**Outils et technologies :** pymongo (`find`), NumPy (`np.array` pour empiler les embeddings en une matrice exploitable par `find_match()`).

### 11. `enregistrer_acces()` — journalisation d'une détection

**Définition** : la journalisation (*logging*) consiste à enregistrer une trace horodatée de chaque événement important — ici, chaque détection (reconnue ou non) — dans un journal consultable, plutôt que de laisser l'information disparaître une fois traitée.

**Rôle** : écrire une trace de chaque détection (connue ou inconnue) dans le journal d'accès, et — si la personne
n'est pas reconnue — créer en plus une fiche en attente d'identification humaine.

**Fonctionnement** : `insert_one(...)` est l'opération d'écriture la plus basique de MongoDB — elle correspond à
un `INSERT` en SQL, mais sans nécessiter de schéma de table prédéfini : chaque document peut en théorie avoir des
champs différents (ici on garde volontairement une structure cohérente pour simplifier les requêtes ultérieures).
C'est le principe même des bases *NoSQL orientées documents* : le schéma est appliqué par la logique applicative
(cette fonction Python), pas imposé rigidement par la base elle-même — flexibilité utile en développement rapide,
au prix d'une responsabilité accrue côté code pour garder les données cohérentes.

**Historique de la notion de « journal d'accès »** : le principe de consigner chronologiquement chaque événement
d'un système remonte au *logging* informatique classique — bien antérieur aux bases NoSQL — et reste le même ici :
chaque document représente un événement immuable, jamais modifié après coup, ce qui permet de reconstruire toute
l'historique d'activité (contrairement à une simple mise à jour de compteur qui perdrait le détail de chaque passage).

**Outils et technologies :** Flask (les arguments viennent de la boucle de détection OpenCV/InsightFace), MongoDB (`insert_one`).

### 12. `charger_stats_du_jour()` — agrégation pour le tableau de bord

**Définition** : une agrégation, en base de données, consiste à combiner plusieurs documents (ici, les détections d'une journée) en une seule structure de résumé exploitable (totaux, moyennes, répartitions) — plutôt que de renvoyer la liste brute à traiter côté interface.

**Rôle** : calculer, à la demande, les chiffres du jour (total, connus, inconnus, première/dernière détection) sans
jamais les stocker à l'avance — ils sont recalculés depuis le journal brut chaque fois que la page dashboard est
visitée.

**Principe** : c'est une **agrégation** au sens des bases de données — transformer un ensemble d'enregistrements
détaillés en quelques statistiques résumées. Ici l'agrégation est faite « à la main » en Python après avoir
rapatrié les documents (`sum(1 for l in ... if ...)` est un simple comptage conditionnel), plutôt qu'avec le
framework d'agrégation natif de MongoDB (`$group`, `$match`, etc., un pipeline inspiré des pipelines Unix). Pour un
volume de données de l'ordre de quelques centaines à quelques milliers de documents par jour, les deux approches
sont équivalentes en pratique ; le pipeline natif Mongo devient préférable si le volume grossit beaucoup, car il
évite de transférer tous les documents bruts vers Python avant de les résumer.

**Outils et technologies :** pymongo (`find`, `.sort()`).

### 13. `calculer_stats_systeme()`

**Définition** : une métrique système est une mesure quantitative de l'état d'une machine ou d'un service (charge CPU, mémoire utilisée, caméras actives) prise à un instant donné, utilisée pour du monitoring (surveillance de bon fonctionnement).

**Rôle** : calculer les 3 indicateurs affichés en haut de la vue temps réel (admin et employé) — état du
système, détections par minute, latence du flux — à partir de données réelles plutôt que de valeurs fixes.

**Fonctionnement** :
- **Détections/min** : nombre de documents dans `Detections` avec une heure dans la dernière minute.
- **Latence** : temps écoulé depuis la dernière frame reçue par `boucle_camera()`, moyenné sur les caméras
  actives (`dernieres_maj_frames`).
- **État** : "Opérationnel" si la latence moyenne est inférieure à 2 secondes, "Dégradé" au-delà, "Hors ligne"
  si aucune caméra n'a encore renvoyé de frame.

**Outils et technologies :** pymongo (`count_documents`), `threading` (lecture protégée de `dernieres_maj_frames` via `verrou_frames`), `datetime`/`timedelta`.

### 14. `charger_inconnus_non_traites()` — file d'attente d'identification

**Définition** : une file d'attente (*queue*) d'identification est simplement l'ensemble des fiches en attente d'une action humaine (ici, attribuer un nom à un visage inconnu) avant de pouvoir être considérées comme traitées.

**Rôle** : récupérer uniquement les fiches d'inconnus pas encore résolues, pour la page `unknowns.html`.

**Fonctionnement** : `{"traite": False}` est un filtre d'égalité — le plus simple des opérateurs de requête
MongoDB. `.sort("date", -1)` trie par ordre décroissant (les plus récentes en premier ; `1` donnerait l'ordre
croissant). C'est l'équivalent conceptuel d'une clause `WHERE traite = false ORDER BY date DESC` en SQL — la
syntaxe diffère (documents vs. clauses textuelles) mais l'opération logique est identique.

**Outils et technologies :** pymongo (`find`, opérateurs `$gte`/`$lt`/`$regex`/`$options`), `re` (expressions régulières, bibliothèque standard), `datetime`/`timedelta`.

### 15. Prétraitement du contraste — CLAHE

**Définition** : le contraste d'une image décrit l'écart entre ses zones claires et sombres ; un éclairage inégal (contre-jour, lumière latérale) réduit ce contraste localement et dégrade la lisibilité des détails du visage pour le réseau de neurones. CLAHE (*Contrast Limited Adaptive Histogram Equalization*) est une technique de traitement d'image qui corrige ce problème localement, tuile par tuile, sans amplifier excessivement le bruit.

**Rôle** : égaliser le contraste de chaque image avant détection/reconnaissance, pour stabiliser les résultats
sous des éclairages très inégaux (webcam en contre-jour, DroidCam/Iriun avec une caméra de téléphone moins
maîtrisée qu'une caméra dédiée).

**Fonctionnement** : CLAHE (*Contrast Limited Adaptive Histogram Equalization*) égalise l'histogramme des
intensités **localement**, par petites tuiles de l'image (ici 8×8), plutôt que sur l'image entière — une
égalisation globale accentuerait le bruit dans les zones déjà bien exposées. Le "Contrast Limited" plafonne
l'amplification du contraste par tuile, pour éviter de sur-amplifier le bruit dans les zones très sombres ou
très claires. Appliqué uniquement sur le canal de **luminance (L)** de l'espace colorimétrique **LAB** — pas
directement sur les canaux Rouge/Vert/Bleu — pour ne pas fausser les couleurs de peau, un indice implicitement
utilisé par le réseau de reconnaissance.

**Origine et histoire** : l'égalisation d'histogramme adaptative a été formalisée dans les années 1980
(Pizer et al., 1987), pour l'imagerie médicale où le contraste local est critique (radiographies). La variante
"Contrast Limited" a été ajoutée peu après pour corriger le sur-bruitage observé sur les images à faible
variation locale. C'est aujourd'hui une fonction standard d'OpenCV (`cv2.createCLAHE`).

**Où c'est appliqué dans ce projet** : sur chaque photo du Dataset au moment de l'extraction des embeddings
(cellule 14), et sur chaque frame de caméra en direct (`boucle_camera()`) — pour que la reconnaissance en
temps réel "voie" le même type d'image que celle utilisée pour construire la base de référence.

**Outils et technologies utilisés** :
- **OpenCV** (`cv2.createCLAHE`, `cv2.cvtColor`, `cv2.split`/`cv2.merge`) — égalisation adaptative de contraste
- **NumPy** (implicite, les images OpenCV sont des tableaux NumPy)

### 16. Data Augmentation du Dataset

**Définition** : l'augmentation de données (*data augmentation*) est une technique qui consiste à générer automatiquement des variantes légèrement modifiées (rotation, luminosité, miroir...) d'une image existante, pour enrichir un jeu de données sans avoir besoin de collecter de nouvelles photos réelles.

**Rôle** : générer plusieurs variantes de chaque photo du Dataset (miroir, luminosité, légère rotation) avant
extraction des embeddings, pour que la base de référence par personne couvre davantage de conditions
(éclairage, angle) qu'une seule photo ne peut en capturer — sans avoir à prendre physiquement plus de photos.

**Fonctionnement** : pour une photo source, 5 variantes supplémentaires sont produites : un miroir horizontal,
deux versions de luminosité (plus sombre / plus claire, via `cv2.convertScaleAbs`), et deux légères rotations
(±10°, via une matrice de rotation `cv2.getRotationMatrix2D` + `cv2.warpAffine`). Chaque variante (CLAHE déjà
appliqué) produit son **propre embedding**, ajouté à la base de référence de la personne — une seule photo
source peut ainsi générer jusqu'à 6 embeddings au lieu d'un seul.

**Origine et histoire** : la data augmentation est une pratique standard en apprentissage automatique depuis
les débuts des réseaux de neurones convolutifs (popularisée notamment par AlexNet, Krizhevsky et al., 2012),
pour compenser un jeu de données limité en générant artificiellement de la variabilité, sans risquer le
surapprentissage (*overfitting*) qu'une simple duplication de données identiques provoquerait.

**Attention** : ici, on n'entraîne pas un modèle — `det_model`/`rec_model` sont déjà pré-entraînés (ArcFace).
L'augmentation sert uniquement à enrichir la **galerie de référence** (les embeddings connus stockés dans
`Persons`), pas à ré-entraîner le réseau lui-même.

**Outils et technologies utilisés** :
- **OpenCV** — `cv2.flip`, `cv2.convertScaleAbs`, `cv2.getRotationMatrix2D`, `cv2.warpAffine`
- **NumPy** (implicite, via les tableaux OpenCV)

### 17. `extract_face_embeddings()` — enrôlement par lot

**Définition** : l'enrôlement biométrique est le processus qui consiste à calculer et enregistrer les embeddings de référence d'une personne à partir de ses photos, pour qu'elle puisse ensuite être reconnue automatiquement. "Par lot" signifie que cette fonction traite toutes les photos de toutes les personnes du dossier `Dataset/` en une seule exécution.

**Rôle** : traiter un dossier `Dataset/<Personne>/*.jpg` pour générer les embeddings de référence de chaque
personne connue, et les insérer en base. C'est l'équivalent de la phase d'**enrôlement** (*enrollment*) en
biométrie — le moment où un système apprend à qui appartient quel visage, avant toute reconnaissance ultérieure.
Ce concept d'enrôlement précède largement le deep learning : il structure historiquement tous les systèmes
biométriques, y compris les plus anciens systèmes d'empreintes digitales (AFIS, *Automated Fingerprint
Identification Systems*, dès les années 1960-1970).

**Fonctionnement** : pour chaque personne (un sous-dossier), la fonction relit chaque photo, détecte le premier
visage trouvé (`bboxes[0]` — hypothèse simplificatrice qu'une photo d'enrôlement ne contient qu'un seul visage
pertinent), calcule son embedding, puis remplace en base tous les anciens embeddings de cette personne
(`delete_many` puis `insert_many`) par les nouveaux — ce mécanisme anti-doublon garantit qu'une ré-exécution sur
les mêmes photos ne fait pas grossir indéfiniment la base.

**Outils et technologies :** OpenCV (`cv2.imread`), InsightFace (`det_model`/`rec_model`), pymongo (`delete_many`, `insert_many`), `glob` et `os` (parcours du système de fichiers).

### 18. Appel désactivé par défaut

**Définition** : une ligne commentée (préfixée par `#`) est ignorée par Python à l'exécution — c'est une façon de garder du code prêt à l'emploi sans qu'il s'exécute automatiquement.

**Rôle** : `extract_face_embeddings()` (cellule précédente) est coûteuse (elle relit et traite toutes les photos du dossier `Dataset/`) et ne doit être relancée que lorsque le contenu de ce dossier change (nouvelle personne, nouvelles photos) — pas à chaque redémarrage du notebook. Cette ligne reste donc commentée par défaut ; il faut retirer le `#` manuellement pour la réexécuter.

**Outils et technologies :** aucun — commentaire Python natif (bibliothèque standard du langage).

### 19. Création de l'application Flask et principe du routage

**Définition** : le routage HTTP est le mécanisme par lequel un serveur web associe chaque URL demandée par un navigateur (ex. `/dashboard`) à la fonction Python chargée d'y répondre. Flask utilise pour cela le décorateur `@app.route(...)`.

**Rôle** : instancier l'objet central de Flask, qui va ensuite recevoir toutes les routes définies dans les
cellules suivantes.

**Principe — le décorateur `@app.route(...)`** : chaque route Flask s'appuie sur les **décorateurs**, une
fonctionnalité du langage Python normalisée par la *PEP 318* en 2003. Un décorateur est une fonction qui
« enveloppe » une autre fonction pour lui ajouter un comportement sans modifier son code — ici, `@app.route("/x")`
enregistre la fonction qui suit dans une table interne à Flask associant l'URL `/x` à cette fonction. Quand une
requête HTTP arrive sur `/x`, Flask consulte cette table et appelle la bonne fonction : c'est le principe du
**routage** URL, central dans tous les frameworks web modernes (popularisé notamment par Ruby on Rails en 2004,
puis largement repris, y compris par Django et Flask).

**Outils et technologies :** Flask (`Flask()`, décorateur `@app.route`) — le cœur du framework web utilisé pour toute l'application.

### 20. `doit_capturer()` — limite les captures à une toutes les 10 secondes

**Définition** : une limite de fréquence (*rate limiting*, ou ici plus précisément un temps de recharge/*cooldown*) est une règle qui empêche une action de se répéter plus souvent qu'un intervalle minimal fixé — ici, pour éviter de capturer une image à chaque frame (plusieurs fois par seconde) pour un même visage.

**Rôle** : avant, chaque frame où un visage restait dans le champ (jusqu'à ~20x/sec) déclenchait
une nouvelle capture (image sauvegardée + entrée journalisée) — cette fonction limite désormais
les captures à une toutes les 10 secondes, par caméra et par identité reconnue.

**Limite connue** : toutes les personnes non reconnues partagent la même clé `"Inconnu"` par
caméra (impossible de les distinguer avant identification) — si deux inconnus différents passent
devant la même caméra à moins de 10 sec d'écart, seul le premier est capturé.

**Outils et technologies :** `threading` (verrou `verrou_captures`), `datetime`/`timedelta`, dictionnaire Python (bibliothèque standard).

### 21. `boucle_camera()`, `demarrer_cameras()` et `generer_frames()` — le pipeline vidéo complet

**Définition** : cette cellule regroupe quatre éléments étroitement liés, qu'on documente ensemble plutôt que de scinder la cellule de code : un verrou (`verrou_modele`, un `threading.Lock`), la fonction `boucle_camera()` (le cœur du système — un thread par caméra, tournant indéfiniment), `demarrer_cameras()` (qui lance un thread par caméra définie dans `CAMERAS`), et `generer_frames()` (qui sert la dernière image prête au navigateur).

**Rôle de `verrou_modele`** : les sessions ONNX Runtime de `det_model`/`rec_model` (cellules 6-7) ne sont pas garanties thread-safe. Comme chaque caméra tourne dans son propre thread (voir `demarrer_cameras()`), deux threads pourraient appeler le même modèle en même temps sans ce verrou — provoquant un échec silencieux de la détection (aucun visage trouvé, sans erreur visible). `with verrou_modele:` garantit qu'un seul thread à la fois utilise les modèles.

**Rôle de `boucle_camera()`** : capturer les images de la caméra choisie en continu, appliquer le prétraitement CLAHE (cellule 13.1), détecter + reconnaître chaque visage, dessiner les rectangles de couleur, journaliser chaque détection (via `enregistrer_acces()`, cellule 11), puis garder la dernière image annotée prête à être servie — indéfiniment, jusqu'à l'arrêt du programme.

**Rôle de `demarrer_cameras()`** : parcourt le dictionnaire `CAMERAS` (cellule 4) et lance un thread `boucle_camera()` par caméra — appelée une seule fois, juste avant `app.run()` (dernière cellule).

**Rôle de `generer_frames()`** : ne capture plus rien elle-même — elle relit simplement la dernière image déjà préparée par `boucle_camera()` (variable partagée `frames_actuelles`) et la transmet au navigateur. Résultat : plusieurs onglets/personnes peuvent regarder la même caméra sans multiplier les connexions physiques, et la détection continue même si personne ne regarde la page.

**Origine et histoire — le format MJPEG en streaming HTTP** : la technique utilisée ici (`multipart/x-mixed-
replace`) a été introduite par Netscape Communications en **1995**, à l'origine pour créer des animations « push »
dans le navigateur Netscape Navigator (le serveur poussait une nouvelle image, qui remplaçait la précédente, sans
que la page ait besoin de se recharger — un ancêtre direct des animations et du contenu dynamique côté serveur).
Cette même technique a ensuite été largement réutilisée par les premières caméras IP et logiciels de webcam dans
les années 2000, précisément parce qu'elle ne nécessite **aucun codec vidéo** : chaque image est indépendamment une
image JPEG classique, envoyée à la suite des autres, séparée par une balise de délimitation (*boundary*). C'est
beaucoup plus simple à mettre en œuvre qu'un vrai flux vidéo compressé (comme le H.264 utilisé par WebRTC, un
protocole bien plus récent, développé par Google et normalisé par le W3C à partir de 2011), au prix d'une
consommation de bande passante beaucoup plus élevée puisqu'aucune compression n'est faite *entre* les images
(pas de compensation de mouvement comme dans un codec vidéo).

**Fonctionnement du protocole** : la structure `b"--frame\r\nContent-Type: image/jpeg\r\n\r\n" + frame_bytes +
b"\r\n"` respecte la norme MIME multipart (RFC 2046) : une ligne de délimitation (`--frame`), un en-tête
décrivant le type de contenu qui suit, une ligne vide obligatoire, puis les octets bruts de l'image, et on
recommence pour l'image suivante. Le navigateur, en recevant un en-tête HTTP `Content-Type: multipart/x-mixed-
replace; boundary=frame` (cellule 18), sait qu'il doit afficher chaque partie à la place de la précédente au fur
et à mesure qu'elle arrive.

**Le mot-clé `yield`** : cette fonction est un **générateur** Python (introduit par la *PEP 255* en 2001). Au lieu
de retourner toutes les images capturées d'un coup (ce qui serait impossible, le flux est infini), elle *suspend*
son exécution à chaque `yield` et la reprend exactement là où elle s'était arrêtée à l'appel suivant — c'est ce
mécanisme qui permet à Flask de renvoyer un flux continu sans jamais charger toute la vidéo en mémoire.

**Outils et technologies :** `threading` (`Lock`, `Thread`), OpenCV (capture, détection, dessin), InsightFace/ONNX Runtime (détection + reconnaissance), CLAHE (cellule 13.1), MongoDB via `enregistrer_acces()`.

### 22. Route `/video_feed/<nom_camera>` — exposer le flux au navigateur

**Définition** : une route (ou *endpoint*) est une URL spécifique que le serveur Flask sait traiter ; celle-ci (`/video_feed/<nom_camera>`) est *dynamique* — la partie `<nom_camera>` de l'URL est une variable capturée et transmise en argument à la fonction Python.

**Rôle** : brancher le générateur précédent sur une URL HTTP consultable par une balise `<img>` dans les pages
`realtime.html` et `vue_users.html`.

**Fonctionnement** : `<nom_camera>` dans le chemin de la route est une **variable d'URL** — Flask capture
automatiquement tout ce qui apparaît à cet endroit et le passe comme argument à la fonction (`CAM-01`,
`CAM-02`, etc.). `mimetype="multipart/x-mixed-replace; boundary=frame"` est l'en-tête HTTP `Content-Type` qui
indique au navigateur le format spécial décrit dans la cellule précédente — sans cet en-tête précis, le navigateur
tenterait d'afficher le flux comme une image fixe unique et échouerait.

**Outils et technologies :** Flask (`Response`, streaming HTTP en continu).

### 23. Route `/` — page de connexion

**Définition** : l'authentification est le processus qui vérifie l'identité d'un utilisateur (ici, via un nom et un mot de passe) avant de lui donner accès à l'application ; cette route sert la page où cette vérification commence.

**Rôle** : servir la page de connexion statique `login.html`.

**Fonctionnement** : `render_template("login.html")` cherche le fichier dans le dossier `templates/` (convention
imposée par Flask) et renvoie son contenu tel quel — aucune variable n'est injectée ici, contrairement aux routes
suivantes.

**Outils et technologies :** Flask (`render_template`), Jinja2 (moteur de templates qui génère le HTML de `login.html`).

### 24. Décorateurs `require_admin` / `require_employe`

**Définition** : un décorateur, en Python, est une fonction qui en enveloppe une autre pour lui ajouter un comportement supplémentaire sans modifier son code interne — ici, vérifier qu'un utilisateur est bien connecté (et avec le bon rôle) avant d'exécuter la vue demandée.

**Rôle** : protéger une route en vérifiant `session["role"]` avant d'exécuter la vue. Si la session ne correspond
pas au rôle attendu, l'utilisateur est renvoyé vers la page de connexion au lieu d'accéder à la page.

**Outils et technologies :** Flask (`session`), `functools.wraps` (bibliothèque standard, préserve les métadonnées de la fonction décorée).

### 25. Routes `/login/admin` et `/login/employe`

**Définition** : une requête POST est le type de requête HTTP utilisé pour envoyer des données au serveur (ici, le contenu d'un formulaire de connexion), par opposition à une requête GET qui ne fait que demander une page.

**Rôle** : traiter la soumission des deux formulaires de `login.html`. L'admin est vérifié par le mot de passe
commun uniquement (`MOT_DE_PASSE_ADMIN`) — n'importe qui connaissant ce mot de passe peut se connecter en tant
qu'admin, le nom saisi ne sert qu'à l'afficher dans l'interface, pas à filtrer l'accès. L'employé, lui, est
vérifié par son nom, qui doit exister dans `db.Persons`. En cas de succès, la session est marquée et
l'utilisateur est redirigé vers son tableau de bord respectif.

**Outils et technologies :** Flask (`request`, `session`, `redirect`), Werkzeug (`check_password_hash`, vérification sécurisée du mot de passe), pymongo.

### 26. Route `/logout`

**Définition** : une session est un mécanisme qui permet au serveur de "se souvenir" qu'un utilisateur est connecté d'une requête à l'autre (via un cookie signé côté navigateur) ; se déconnecter consiste à effacer cette information.

**Rôle** : vider la session (déconnexion), pour les deux rôles.

**Outils et technologies :** Flask (`session`).

### 27. `charger_detections_semaine()` — agrégation par jour

**Définition** : une agrégation par jour consiste à compter, pour chacun des 7 derniers jours, le nombre de documents `Detections` correspondants — un total par jour, plutôt que la liste brute.

**Rôle** : préparer les données du graphique "Détections par jour" du tableau de bord, en comptant les détections des 7 derniers jours (aujourd'hui inclus) jour par jour.

**Fonctionnement** : une boucle sur les 7 derniers jours (`datetime.now() - timedelta(days=i)`), et pour chacun, `db.Detections.count_documents({"date": date_str})` — une requête de comptage par jour plutôt qu'une seule requête sur toute la période, pour obtenir directement la répartition jour par jour attendue par le graphique.

**Outils et technologies :** pymongo (`count_documents`), `datetime`/`timedelta`.

### 28. Route `/dashboard`

**Définition** : un tableau de bord (*dashboard*) est une page qui résume visuellement l'état global d'un système (statistiques, graphiques, indicateurs), pour une lecture rapide sans avoir à consulter le détail brut des données.

**Rôle** : relier la fonction d'agrégation (cellule 12) au template `dashboard.html`.

**Fonctionnement — le moteur de templates Jinja2** : `render_template("dashboard.html", stats=stats)` transmet le
dictionnaire `stats` au moteur Jinja2, qui remplacera dans le HTML chaque `{{ stats.total }}` (par exemple) par sa
valeur réelle au moment du rendu. Jinja2 a été créé par Armin Ronacher (le même auteur que Flask) en 2008, en
s'inspirant du système de templates de Django (2005), lui-même héritier d'une longue tradition de moteurs de
templates web remontant aux *Server Side Includes* (SSI) du serveur NCSA HTTPd dans les années 1990.

**Outils et technologies :** Flask (`render_template`), pymongo, `psutil` (métriques CPU/mémoire), `shutil` (espace disque).

### 29. Route `/realtime`

**Définition** : le *polling* est une technique où le navigateur interroge le serveur à intervalles réguliers (ici via JavaScript, toutes les quelques secondes) pour vérifier si de nouvelles données sont disponibles, simulant ainsi une mise à jour "en direct" sans rechargement complet de la page.

**Rôle** : servir la page de supervision temps réel (vue administrateur). Elle ne transmet aucune donnée calculée
pour l'instant — le flux vidéo est chargé séparément par le navigateur via `/video_feed/<nom_camera>` (cellule 17),
appelé directement depuis la balise `<img>` du template, indépendamment de cette route.

**Outils et technologies :** Flask (`render_template`).

### 30. Route `/api/dernieres_detections` — API JSON pour l'auto-actualisation

**Définition** : une API JSON est une route qui, au lieu de renvoyer une page HTML complète, renvoie uniquement des données brutes au format JSON — destinées à être lues et affichées par du JavaScript côté navigateur, sans recharger toute la page.

**Rôle** : permettre au JavaScript de `realtime.html` d'interroger le serveur toutes les quelques secondes et de mettre à jour uniquement la liste des dernières détections, sans recharger toute la page — ce qui couperait le flux vidéo MJPEG en cours (cellule 16).

**Fonctionnement** : réutilise `charger_stats_du_jour()` (cellule 12), puis reconstruit manuellement chaque entrée en un dictionnaire simple (`id`, `nom`, `statut`, `heure`, `camera`, `score_detection`, `image`) — nécessaire car le document MongoDB brut contient un `_id` de type `ObjectId`, que `jsonify()` ne sait pas convertir directement en JSON.

**Outils et technologies :** Flask (`jsonify`, qui sérialise un dictionnaire Python en réponse JSON avec le bon en-tête `Content-Type`).

### 31. Route `/vue_users`

**Définition** : une "vue" désigne ici la page qui liste les comptes utilisateurs (administrateurs/employés) existants dans l'application, distincte des personnes reconnues par la caméra (qui, elles, n'ont pas de compte de connexion).

**Rôle** : la même logique que `/realtime`, mais pour la vue destinée aux employés (moins d'informations
administratives affichées). `nom_camera="CAM-01"` est transmis en dur pour l'instant — une seule caméra affichée
par page ; un sélecteur multi-caméras pourra être ajouté plus tard si nécessaire.

**Outils et technologies :** Flask (`render_template`), pymongo.

### 32. `charger_statistiques()` — agrégations pour la page Statistiques

**Définition** : une jointure, en base de données, consiste à combiner des informations provenant de deux collections différentes (ici, les détections du jour et les fiches `Persons`) en s'appuyant sur un champ commun (le nom), pour enrichir le résultat final.

**Rôle** : calculer trois choses pour `statistics.html` : la répartition des détections par heure, le taux global
de reconnaissance, et le détail par personne (avec rôle/département).

**Principe — l'histogramme par heure** : `presence_par_heure` est un **histogramme** au sens statistique classique
(le terme a été proposé par Karl Pearson en 1895) : on découpe la journée en 24 catégories (les heures), et on
compte combien de détections tombent dans chaque catégorie. C'est la structure de données la plus simple pour
visualiser une distribution de fréquence dans le temps — exactement ce qu'affichent les barres du graphique
« Présence par heure » de votre page HTML.

**Le taux de reconnaissance** est un simple ratio (`connus / total × 100`), une proportion — rien de plus qu'une
règle de trois, mais c'est la métrique la plus lisible pour un humain qui veut juger la performance globale du
système sans lire un journal détaillé.

**La jointure manuelle** (`db.Persons.find({"nom": {"$in": noms}}, ...)`) : MongoDB, en tant que base
orientée documents, n'a historiquement pas de jointure native aussi simple qu'un `JOIN` SQL (les données sont
plutôt censées être dénormalisées, c'est-à-dire dupliquées dans chaque document pour éviter d'avoir à joindre).
Ici on fait le lien « à la main » entre deux collections (`Detections` et `Persons`) en récupérant les
fiches correspondantes via l'opérateur `$in` (« la valeur du champ `nom` doit être l'une de ces valeurs de la
liste »), puis en construisant un dictionnaire Python pour un accès rapide (`infos_par_nom.get(nom, {})`) — une
solution simple, adaptée au faible volume de personnes différentes par jour dans ce contexte.

**Outils et technologies :** pymongo (opérateur `$in` pour la jointure avec `Persons`).

### 33. Route `/statistics`

**Définition** : cette route sert simplement la page qui affiche les statistiques déjà calculées par `charger_statistiques()` (section précédente) — elle ne fait aucun calcul elle-même.

**Rôle** : relier `charger_statistiques()` au template `statistics.html`, selon le même principe que la route
`/dashboard` (cellule 19).

**Outils et technologies :** Flask (`render_template`).

### 34. Route `/unknowns`

**Définition** : cette route sert la page listant les visages détectés mais non encore identifiés (voir section 13, `charger_inconnus_non_traites()`), avec ses éventuels filtres de période/recherche.

**Rôle** : afficher la liste des inconnus en attente d'identification.

**Pourquoi convertir `_id` en chaîne ?** Chaque document MongoDB possède un champ `_id` de type **ObjectId** —
un identifiant binaire de 12 octets (4 octets d'horodatage + 5 octets aléatoires générés une fois par processus +
3 octets de compteur incrémental), conçu par les ingénieurs de MongoDB pour garantir l'unicité *sans coordination
centrale*, contrairement aux identifiants auto-incrémentés des bases SQL classiques qui nécessitent un point de
synchronisation unique. Le moteur de templates Jinja2 ne sait afficher que du texte simple : `str(doc["_id"])`
convertit cet identifiant binaire en sa représentation textuelle hexadécimale, utilisable dans un champ caché du
formulaire HTML (voir cellule 27, où cette valeur revient sous forme de texte).

**Outils et technologies :** Flask (`request.args`, `render_template`).

### 35. `identifier_inconnu()` — transformer un inconnu en personne connue

**Définition** : "transformer un inconnu en personne connue" signifie déplacer les données d'une fiche `UnknownPersons` (en attente) vers la collection `Persons` (base des personnes reconnues), une fois qu'un administrateur lui a attribué un nom.

**Rôle** : quand un administrateur identifie manuellement un inconnu via le formulaire, cette fonction relit
l'image sauvegardée, y recalcule un embedding, l'ajoute à `personnes_connues`, et marque la fiche comme traitée.

**Pourquoi recalculer l'embedding plutôt que le réutiliser ?** Au moment de la détection initiale (cellule 16), on
a choisi de ne sauvegarder que l'*image* du visage, pas son embedding — un choix qui simplifie le flux principal
(pas besoin de conserver un vecteur de 512 flottants en attente), au prix de devoir refaire tourner les deux
modèles (détection + reconnaissance) une seconde fois sur l'image, une fois l'identité confirmée. Pour le faible
volume de fiches à traiter manuellement, ce recalcul reste négligeable en temps de calcul.

**`ObjectId(detection_id)`** reconstruit l'identifiant binaire MongoDB à partir de sa représentation textuelle
reçue du formulaire — l'opération inverse exacte de `str(doc["_id"])` faite en cellule 24.

**Outils et technologies :** OpenCV (`cv2.imread`), InsightFace (`det_model`/`rec_model`), pymongo (`insert_one`, `update_one`), `bson.ObjectId` (conversion d'identifiant MongoDB).

### 36. Route `/identifier` — traitement du formulaire

**Définition** : cette route traite la soumission du formulaire d'identification (voir `unknowns.html`) — elle réceptionne les données saisies par l'administrateur et appelle `identifier_inconnu()` (section précédente) pour les enregistrer.

**Rôle** : recevoir la soumission du formulaire d'identification (méthode HTTP `POST`), appeler
`identifier_inconnu()`, puis recharger la base d'embeddings en mémoire pour que la reconnaissance en direct
(`/video_feed`) prenne immédiatement en compte la nouvelle personne.

**GET vs POST** : ces deux verbes du protocole HTTP (normalisés dès la version 1.0 de HTTP, 1996) ont des
sémantiques différentes — `GET` est censé être *sans effet de bord* (consulter une page), `POST` est destiné à
*modifier un état* (ici, créer une nouvelle personne connue). Utiliser `POST` pour ce formulaire respecte cette
convention et évite, par exemple, qu'un navigateur ne réexécute accidentellement l'identification simplement en
rafraîchissant la page.

**Le mot-clé `global`** : sans lui, l'affectation `known_embeddings, known_names = ...` à l'intérieur de la
fonction créerait des variables *locales* à cette fonction, sans effet sur les variables du même nom utilisées par
`generer_frames()` ailleurs dans le notebook — une subtilité classique de la portée des variables en Python, où
une affectation dans une fonction est locale par défaut, sauf déclaration explicite du contraire.

**Outils et technologies :** Flask (`request.form`), mot-clé `global` (portée des variables en Python, bibliothèque standard du langage).

### 37. Route `/personnes_connues`

**Définition** : cette route sert la page listant l'ensemble des personnes déjà reconnues par le système (contenu de la collection `Persons`, dédupliqué par nom), avec ses éventuels filtres.

**Rôle** : afficher le tableau des personnes déjà identifiées (`personnesconnues.html`), avec leur poste, département et statut. Suit le même principe que `/dashboard` (cellule 19) : une fonction d'agrégation dédiée alimente le template.

**Outils et technologies :** pymongo (`distinct`, `find_one`), tri Python natif (`sorted`/`.sort()`).

### 38. Route `/modifier_personne` — édition du rôle/département d'un employé

**Définition** : un formulaire d'édition est un formulaire HTML pré-rempli avec les valeurs actuelles d'un enregistrement, qui envoie au serveur uniquement les champs modifiables — ici ouvert depuis `personnesconnues.html`.

**Rôle** : traiter la soumission de ce formulaire, pour corriger le rôle/poste et le département d'une personne déjà enrôlée.

**Pourquoi `update_many` et pas `update_one` ?** `Persons` contient un document par PHOTO d'enrôlement (un par embedding), donc plusieurs dizaines de documents pour une même personne (voir `extract_face_embeddings`, cellule 14). Modifier uniquement le document ciblé par son `_id` laisserait le rôle/département incohérent entre les différentes photos de la même personne ; `update_many` sur le champ `nom` met à jour toutes ses fiches en une seule fois.

**Outils et technologies :** Flask (`request.form`, `redirect`, `url_for`), MongoDB (`update_many`, opérateur `$set`).

### 39. Route `/detections`

**Définition** : cette route sert la page listant l'historique complet des détections (reconnues et refusées), avec ses éventuels filtres de période/recherche — à ne pas confondre avec la page "Inconnus", qui ne liste que les visages non identifiés.

**Rôle** : afficher la liste complète des dernières détections (`dernieres_detections.html`), réutilisant `charger_stats_du_jour()` (cellule 12) pour la liste `dernieres` déjà calculée pour le dashboard.

**Outils et technologies :** Flask (`request.args`), pymongo (opérateurs `$gte`/`$regex`/`$or`), `re`.

### 40. Stockage des administrateurs (collection Users)

**Définition** : la collection `Users` stocke les comptes ayant le droit de se connecter à l'interface d'administration — à ne pas confondre avec `Persons`, qui stocke les personnes reconnues par la caméra (qui n'ont pas de compte).

**Rôle** : gérer les comptes admin dans `Users`, avec un **mot de passe unique partagé par tous les administrateurs**
(pas un mot de passe par personne). Chaque admin a son propre document (juste un nom, pour l'identifier), et un
document séparé (`{"type": "mot_de_passe_admin", ...}`) stocke le hash du mot de passe commun. Se connecter demande
donc un nom qui existe bien parmi les admins **et** le mot de passe partagé.

**Important** : au tout premier lancement, un email admin par défaut et un mot de passe commun par défaut
(`00000000`) sont créés automatiquement — à changer immédiatement depuis la page Paramètres.

**Outils et technologies :** pymongo, Werkzeug (`generate_password_hash`, `check_password_hash`).

### 41. Route `/parametres`

**Définition** : cette route sert la page de paramètres généraux de l'application (gestion des caméras, mot de passe partagé, etc.).

**Rôle** : afficher la page Paramètres (`parametres.html`), avec la liste actuelle des caméras.

**Outils et technologies :** Flask (`render_template`).

### 42. Route `/parametres/mot_de_passe`

**Définition** : cette route traite le changement du mot de passe partagé par tous les administrateurs.

**Rôle** : traiter le formulaire de changement de mot de passe. Vérifie l'ancien mot de passe avec `check_password_hash`, exige une confirmation, puis réécrit `data/admin.json`.

**Outils et technologies :** Werkzeug (`check_password_hash`, `generate_password_hash`), `json` (bibliothèque standard, fichier de configuration).

### 43. Route `/parametres/camera/supprimer/<nom_camera>`

**Définition** : cette route traite la suppression d'une caméra de la configuration (`data/cameras.json`), identifiée par son nom dans l'URL.

**Rôle** : retirer une caméra de `CAMERAS` et persister le changement.

**Outils et technologies :** `json` et `os` (bibliothèque standard, réécriture de `data/cameras.json`).

### 44. Chargement initial des embeddings connus

**Définition** : ce bloc n'est pas une fonction mais du code exécuté une seule fois, au démarrage du programme, qui charge en mémoire (variables globales) les embeddings connus, pour que les boucles de détection n'aient pas à interroger MongoDB à chaque frame.

**Rôle** : charger une seule fois, avant de démarrer le serveur, les embeddings depuis MongoDB — ce sont ces
variables globales que `generer_frames()` (cellule 16) et la route `/identifier` (cellule 26) utilisent et mettent
à jour ensuite.

**Outils et technologies :** pymongo, NumPy.

### 45. Lancement du serveur Flask

**Définition** : lancer le serveur (`app.run(...)`) démarre le serveur de développement WSGI intégré à Flask/Werkzeug, qui se met à écouter les requêtes HTTP entrantes sur le réseau — c'est la toute dernière étape, celle qui rend l'application effectivement accessible.

**Rôle** : démarrer le serveur de développement intégré à Flask, qui écoute désormais les requêtes HTTP.

**Fonctionnement des paramètres** :
- `host="0.0.0.0"` : écoute sur toutes les interfaces réseau de la machine (pas seulement `localhost`), ce qui
  permet d'accéder au serveur depuis un autre appareil du réseau local (utile pour tester depuis un téléphone,
  par exemple, en plus de la connexion DroidCam elle-même)
- `debug=True` : active le mode debug de Flask, qui affiche des pages d'erreur détaillées en cas d'exception
- `use_reloader=False` : **indispensable dans un notebook**. Le *reloader* de Flask, en temps normal, surveille
  les fichiers source et relance tout le processus Python automatiquement à chaque modification — un mécanisme
  incompatible avec un kernel Jupyter, qu'il interromprait de façon incontrôlée.
- `threaded=True` : **indispensable dès qu'il y a plus d'une caméra**. Sans ce paramètre, le serveur de
  développement ne traite qu'une seule requête à la fois. Or chaque flux vidéo (`/video_feed/<camera>`) est une
  réponse *infinie* (le générateur `generer_frames()` ne se termine jamais tant que la caméra est branchée) — la
  première caméra qui se connecte monopoliserait alors indéfiniment l'unique thread disponible, et toute autre
  requête (une deuxième caméra, ou même une simple navigation) resterait bloquée en attente.

**Origine — WSGI** : le serveur intégré de Flask parle le protocole **WSGI** (*Web Server Gateway Interface*,
normalisé par la *PEP 3333* en 2010), qui définit une interface standard entre un serveur web et une application
Python — n'importe quelle application respectant WSGI peut tourner derrière n'importe quel serveur compatible.
WSGI a succédé au protocole **CGI** (*Common Gateway Interface*, créé en 1993 par le NCSA, l'un des tout premiers
standards ayant permis à des scripts serveur de générer des pages web dynamiques). Ce serveur de développement
n'est volontairement pas conçu pour un usage en production à forte charge — Flask le rappelle d'ailleurs dans ses
propres avertissements de démarrage ; des serveurs comme Gunicorn ou uWSGI prendraient le relais pour un déploiement
réel, mais restent hors du périmètre de ce projet académique.

**Outils et technologies :** Flask (`app.run`), Werkzeug (serveur de développement WSGI sous-jacent), `threading` (démarrage des boucles caméra en parallèle du serveur web).